In [21]:
import sys
sys.path.append("../src")

import importlib
import pandas as pd

import metrics
import backtesting

importlib.reload(metrics)
importlib.reload(backtesting)

WINDOW = 3 * 252
STEP = 21
TRANSACTION_COST = 0.001
VOL_TARGET = 0.10
MAX_LEVERAGE = 2.0
FUNDING_SPREAD = 0.01

prices = pd.read_csv("../data/etf_prices_raw.csv", index_col="Date", parse_dates=True)
returns = metrics.compute_returns(prices)

rf_rate = pd.read_csv("../data/risk_free_rate.csv", index_col="Date", parse_dates=True)["risk_free_rate"]
rf_rate = pd.to_numeric(rf_rate, errors="coerce").ffill()

annual_rf = rf_rate / 100
risk_free_returns = (1 + annual_rf) ** (1 / 252) - 1
risk_free_returns = risk_free_returns.reindex(returns.index).ffill()

groups = {
    "equities": ["SPY", "QQQ", "EWU", "EZU", "EEM"],
    "bonds": ["IEF", "SHY"],
    "alternatives": ["GLD", "VNQ"]
}

strategies = {
    "Equal Weight": (backtesting.equal_strategy, None),
    "Inverse Vol": (backtesting.risk_strategy, None),
    "Max Geo": (backtesting.max_geo_strategy, None),
    "Max Sharpe": (backtesting.max_sharpe_strategy, None),
    "Carver": (backtesting.carver_handcrafting_strategy, groups),
    "Hierarchical Inv Vol": (backtesting.inv_handcrafting_strategy, groups),
    "SPY": (backtesting.spy_strategy, None),
    "60/40": (backtesting.sixty_forty_strategy, None),
    "Tech": (backtesting.tech_strategy, None)
}

In [22]:
def run_robustness_returns(window=WINDOW, step=STEP, vol_target=VOL_TARGET,
                           max_leverage=MAX_LEVERAGE,
                           transaction_cost=TRANSACTION_COST,
                           funding_spread=FUNDING_SPREAD):

    portfolio_returns = {}

    for name, (strategy, strategy_groups) in strategies.items():
        port_ret, _, _, _, _ = backtesting.backtest(
            returns,
            window=window,
            step=step,
            allocation_func=strategy,
            groups=strategy_groups,
            vol_target=vol_target,
            max_leverage=max_leverage,
            risk_free_returns=risk_free_returns,
            transaction_cost=transaction_cost,
            funding_spread=funding_spread
        )

        portfolio_returns[name] = port_ret

    return pd.DataFrame(portfolio_returns)

In [23]:
# Window

windows = {"2Y": 2 * 252, "3Y": 3 * 252,"5Y": 5 * 252}

window_returns = {}

for label, window in windows.items():
    window_returns[label] = run_robustness_returns(window=window)

common_index = (
    window_returns["2Y"].index
    .intersection(window_returns["3Y"].index)
    .intersection(window_returns["5Y"].index)
)

for label, port_returns in window_returns.items():
    aligned = port_returns.loc[common_index]
    results = []

    for strategy in aligned.columns:
        port_ret = aligned[strategy]

        results.append({
            "Strategy": strategy,
            "Annual Return": metrics.annualize_returns(port_ret),
            "Annual Volatility": metrics.annualize_volatility(port_ret),
            "Sharpe Ratio": metrics.sharpe_ratio(port_ret, risk_free_returns),
            "Drawdown": metrics.drawdown(port_ret)
        })

    result_table = pd.DataFrame(results)

    print(label)
    display(result_table)

2Y


,Strategy,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
0,Equal Weight,0.064154,0.104350,0.504386,-0.248580
1,Inverse Vol,0.048901,0.080594,0.446425,-0.248562
2,Max Geo,0.074855,0.110828,0.571490,-0.262939
3,Max Sharpe,0.065416,0.090832,0.577896,-0.206872
4,Carver,0.037092,0.061082,0.380911,-0.217831
5,Hierarchical Inv Vol,0.040507,0.065413,0.410166,-0.223019
6,SPY,0.083518,0.105533,0.670693,-0.217877
7,60/40,0.092647,0.105191,0.752253,-0.210350
8,Tech,0.100119,0.103394,0.829485,-0.149707


3Y


,Strategy,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
0,Equal Weight,0.066732,0.099936,0.546446,-0.270695
1,Inverse Vol,0.050142,0.077598,0.475863,-0.239383
2,Max Geo,0.093019,0.102307,0.773997,-0.222613
3,Max Sharpe,0.074405,0.086549,0.699372,-0.225831
4,Carver,0.038681,0.061294,0.404805,-0.212516
5,Hierarchical Inv Vol,0.041735,0.065341,0.428619,-0.215909
6,SPY,0.086331,0.099561,0.730864,-0.244055
7,60/40,0.094442,0.100017,0.802322,-0.234723
8,Tech,0.101255,0.099649,0.867223,-0.161984


5Y


,Strategy,Annual Return,Annual Volatility,Sharpe Ratio,Drawdown
0,Equal Weight,0.065321,0.094608,0.557689,-0.252080
1,Inverse Vol,0.051387,0.077152,0.493495,-0.243406
2,Max Geo,0.080025,0.099595,0.672221,-0.217016
3,Max Sharpe,0.071013,0.086186,0.665261,-0.269880
4,Carver,0.037443,0.060351,0.390379,-0.214199
5,Hierarchical Inv Vol,0.040907,0.064948,0.418549,-0.218017
6,SPY,0.083466,0.096807,0.721574,-0.245975
7,60/40,0.091292,0.098468,0.784112,-0.235671
8,Tech,0.100112,0.099223,0.860055,-0.167825


In [ ]:
# Step

steps = {"Weekly": 5,"Monthly": 21, "Quarterly": 63}

step_returns = {}

for label, step in steps.items():
    step_returns[label] = run_robustness_returns(step=step)

common_index = (
    step_returns["Weekly"].index
    .intersection(step_returns["Monthly"].index)
    .intersection(step_returns["Quarterly"].index)
)

for label, port_returns in step_returns.items():
    aligned = port_returns.loc[common_index]
    results = []

    for strategy in aligned.columns:
        port_ret = aligned[strategy]

        results.append({
            "Strategy": strategy,
            "Annual Return": metrics.annualize_returns(port_ret),
            "Annual Volatility": metrics.annualize_volatility(port_ret),
            "Sharpe Ratio": metrics.sharpe_ratio(port_ret, risk_free_returns),
            "Drawdown": metrics.drawdown(port_ret)
        })

    result_table = pd.DataFrame(results)

    print(label)
    display(result_table)

In [ ]:
# Vol target

vol_targets = {"5%": 0.05,"10%": 0.10,"15%": 0.15}

vol_target_returns = {}

for label, vol_target in vol_targets.items():
    vol_target_returns[label] = run_robustness_returns(vol_target=vol_target)

common_index = (
    vol_target_returns["5%"].index
    .intersection(vol_target_returns["10%"].index)
    .intersection(vol_target_returns["15%"].index)
)

for label, port_returns in vol_target_returns.items():
    aligned = port_returns.loc[common_index]
    results = []

    for strategy in aligned.columns:
        port_ret = aligned[strategy]

        results.append({
            "Strategy": strategy,
            "Annual Return": metrics.annualize_returns(port_ret),
            "Annual Volatility": metrics.annualize_volatility(port_ret),
            "Sharpe Ratio": metrics.sharpe_ratio(port_ret, risk_free_returns),
            "Drawdown": metrics.drawdown(port_ret)
        })

    result_table = pd.DataFrame(results)

    print(label)
    display(result_table)

In [ ]:
# Max leverage

leverage_caps = {"1x": 1.0,"1.5x": 1.5,"2x": 2.0}

leverage_returns = {}

for label, max_leverage in leverage_caps.items():
    leverage_returns[label] = run_robustness_returns(max_leverage=max_leverage)

common_index = (
    leverage_returns["1.0x"].index
    .intersection(leverage_returns["1.5x"].index)
    .intersection(leverage_returns["2.0x"].index)
)

for label, port_returns in leverage_returns.items():
    aligned = port_returns.loc[common_index]
    results = []

    for strategy in aligned.columns:
        port_ret = aligned[strategy]

        results.append({
            "Strategy": strategy,
            "Annual Return": metrics.annualize_returns(port_ret),
            "Annual Volatility": metrics.annualize_volatility(port_ret),
            "Sharpe Ratio": metrics.sharpe_ratio(port_ret, risk_free_returns),
            "Drawdown": metrics.drawdown(port_ret)
        })

    result_table = pd.DataFrame(results)

    print(label)
    display(result_table)